# NYC Mobility - Business Analytics

## Analytics scope

This compiled notebook creates the three business-ready views used by the Analytics Dashboard. The modular SQL remains in `src/05_analytics/`, while this notebook provides the documented execution sequence used by the Databricks job.

All views use accepted Gold fact rows where `dq_out_of_range_datetime = FALSE`. Analytics creation remains separate from Gold modeling and from Analytics validation.

## Taxi Demand by Day, Hour, and Zone

Creates one row per pickup date, pickup hour, and pickup Taxi Zone for demand ranking and trip summaries.

The SQL uses governed Gold keys and measures and publishes a reusable dashboard view.

In [ ]:
CREATE OR REPLACE VIEW `ftw-week-08`.`03_gold`.analytics_taxi_demand AS
SELECT
    TO_DATE(f.pickup_datetime) AS pickup_date,
    DATE_FORMAT(f.pickup_datetime, 'EEEE') AS pickup_day_name,
    f.pickup_taxi_zone_key,
    t.hour_24 AS pickup_hour_24,
    t.hour_label AS pickup_hour_label,

    z.zone_name AS pickup_zone,
    z.borough AS pickup_borough,

    COUNT(*) AS trip_volume,
    ROUND(AVG(f.trip_duration_minutes), 2) AS avg_trip_duration_minutes,
    ROUND(AVG(f.trip_distance), 2) AS avg_trip_distance_miles,
    ROUND(SUM(f.total_amount), 2) AS total_trip_amount

FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f

LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS z
    ON f.pickup_taxi_zone_key = z.taxi_zone_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS t
    ON f.pickup_time_key = t.time_key

WHERE f.dq_out_of_range_datetime = FALSE

GROUP BY
    TO_DATE(f.pickup_datetime),
    DATE_FORMAT(f.pickup_datetime, 'EEEE'),
    f.pickup_taxi_zone_key,
    t.hour_24,
    t.hour_label,
    z.zone_name,
    z.borough

ORDER BY trip_volume DESC;


## Weather and Green Taxi Trip Behavior

Creates one row per weather condition and weather code for descriptive association analysis using trip-weighted weather measures.

The SQL uses governed Gold keys and measures and publishes a reusable dashboard view.

In [ ]:
CREATE OR REPLACE VIEW `ftw-week-08`.`03_gold`.analytics_weather_behavior AS
WITH trip_weather AS (
    SELECT
        f.trip_key,
        f.trip_duration_minutes,
        f.trip_distance,
        f.fare_amount,
        f.total_amount,

        w.weather_code,
        w.temperature_2m,
        w.precipitation,
        w.rain,
        w.snowfall,
        w.wind_speed_10m,

        -- Exact WMO weather-code classification (matches area_mobility_patterns.sql):
        CASE
            WHEN w.weather_code = 0 THEN 'Clear'
            WHEN w.weather_code IN (1, 2, 3) THEN 'Cloudy'
            WHEN w.weather_code IN (45, 48) THEN 'Fog'
            WHEN w.weather_code IN (51, 53, 55, 56, 57) THEN 'Drizzle'
            WHEN w.weather_code IN (61, 63, 65, 66, 67, 80, 81, 82) THEN 'Rain'
            WHEN w.weather_code IN (71, 73, 75, 77, 85, 86) THEN 'Snow'
            WHEN w.weather_code IN (95, 96, 99) THEN 'Thunderstorm'
            ELSE 'Other / Unknown'
        END AS weather_condition

    FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f

    INNER JOIN `ftw-week-08`.`03_gold`.dim_weather_hour AS w
        ON f.pickup_weather_hour_key = w.weather_hour_key

    WHERE f.dq_out_of_range_datetime = FALSE
)

SELECT
    weather_condition,
    weather_code,

    COUNT(*) AS trip_volume,
    ROUND(AVG(trip_duration_minutes), 2) AS avg_trip_duration_minutes,
    ROUND(AVG(trip_distance), 2) AS avg_trip_distance_miles,
    ROUND(AVG(fare_amount), 2) AS avg_fare_amount,
    ROUND(SUM(total_amount), 2) AS total_trip_amount,

    ROUND(AVG(temperature_2m), 2) AS trip_weighted_avg_temperature_c,
    ROUND(AVG(precipitation), 2) AS trip_weighted_avg_precipitation_mm,
    ROUND(AVG(rain), 2) AS trip_weighted_avg_rain_mm,
    ROUND(AVG(snowfall), 2) AS trip_weighted_avg_snowfall_cm,
    ROUND(AVG(wind_speed_10m), 2) AS trip_weighted_avg_wind_speed_10m

FROM trip_weather

GROUP BY
    weather_condition,
    weather_code

ORDER BY trip_volume DESC;


## Area Mobility Patterns

Creates one row per Taxi Zone with pickup or drop-off activity, including trip measures, peak pickup hour, and weather-related pickup patterns.

The SQL uses governed Gold keys and measures and publishes a reusable dashboard view.

In [ ]:
-- Pickup-specific metrics remain NULL when a zone has no pickups.
-- Depends on: Gold fact_green_taxi_trip, dim_taxi_zone, dim_time, and dim_weather_hour.
-- Why: Supports comparison of mobility patterns across NYC taxi zones.
-- Note: High activity or trip amounts do not automatically indicate profitability, revenue, or underserved areas.
--       (avg_trip_duration_minutes, avg_trip_distance_miles, total_trip_amount, avg_trip_amount,
--       adverse_weather_pickups, clear_or_cloudy_pickups, adverse_weather_pickup_share_pct, and
--       peak_pickup_hour_*) are NULL rather than 0 — there's no pickup trip to compute them from.

CREATE OR REPLACE VIEW `ftw-week-08`.`03_gold`.analytics_area_mobility_patterns AS
WITH pickup_metrics AS (
    SELECT
        f.pickup_taxi_zone_key AS taxi_zone_key,

        COUNT(*) AS pickup_trip_volume, -- Number of accepted Green Taxi trip records originating from the zone
        COUNT(DISTINCT TO_DATE(f.pickup_datetime)) AS active_pickup_days, -- Number of distinct calendar dates with at least one pickup

        ROUND(AVG(f.trip_duration_minutes), 2) AS avg_trip_duration_minutes,
        ROUND(AVG(f.trip_distance), 2) AS avg_trip_distance_miles,
        ROUND(SUM(f.total_amount), 2) AS total_trip_amount,
        ROUND(AVG(f.total_amount), 2) AS avg_trip_amount,

        -- Count of pickups whose weather hour falls under an adverse WMO weather code
        -- (exact codes, matching weather_behavior.sql's classification):
        --   51, 53, 55, 56, 57 = Drizzle
        --   61, 63, 65, 66, 67, 80, 81, 82 = Rain
        --   71, 73, 75, 77, 85, 86 = Snow
        --   95, 96, 99 = Thunderstorm
        SUM(
            CASE
                WHEN w.weather_code IN (
                    51, 53, 55, 56, 57,
                    61, 63, 65, 66, 67,
                    71, 73, 75, 77,
                    80, 81, 82,
                    85, 86,
                    95, 96, 99
                )
                THEN 1
                ELSE 0
            END
        ) AS adverse_weather_pickups,

        -- Count pickups occurring under clear/cloudy weather codes (WMO 0-3).
        SUM(
            CASE
                WHEN w.weather_code IN (0, 1, 2, 3)
                THEN 1
                ELSE 0
            END
        ) AS clear_or_cloudy_pickups

    FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f

    LEFT JOIN `ftw-week-08`.`03_gold`.dim_weather_hour AS w
        ON f.pickup_weather_hour_key = w.weather_hour_key

    WHERE f.dq_out_of_range_datetime = FALSE
    GROUP BY f.pickup_taxi_zone_key
),

dropoff_metrics AS (
    SELECT
        dropoff_taxi_zone_key AS taxi_zone_key,
        COUNT(*) AS dropoff_trip_volume

    FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip
    WHERE dq_out_of_range_datetime = FALSE
    GROUP BY dropoff_taxi_zone_key
),

zone_hour_counts AS (
    SELECT
        f.pickup_taxi_zone_key AS taxi_zone_key,
        t.hour_24 AS pickup_hour_24,
        t.hour_label AS pickup_hour_label,
        COUNT(*) AS trip_volume

    FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f

    LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS t
        ON f.pickup_time_key = t.time_key

    WHERE f.dq_out_of_range_datetime = FALSE

    GROUP BY
        f.pickup_taxi_zone_key,
        t.hour_24,
        t.hour_label
),

peak_hour AS (
    SELECT
        taxi_zone_key,
        pickup_hour_24 AS peak_pickup_hour_24,
        pickup_hour_label AS peak_pickup_hour_label,
        trip_volume AS peak_hour_trip_volume

    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY taxi_zone_key
                ORDER BY
                    trip_volume DESC,
                    pickup_hour_24 ASC NULLS LAST
            ) AS hour_rank
        FROM zone_hour_counts
    )
    WHERE hour_rank = 1
)

SELECT
    COALESCE(
        p.taxi_zone_key,
        d.taxi_zone_key
    ) AS taxi_zone_key,
    z.zone_name,
    z.borough,
    z.service_zone,

    COALESCE(p.pickup_trip_volume, 0) AS pickup_trip_volume,
    COALESCE(d.dropoff_trip_volume, 0) AS dropoff_trip_volume,
    COALESCE(p.active_pickup_days, 0) AS active_pickup_days,

    p.avg_trip_duration_minutes,
    p.avg_trip_distance_miles,
    p.total_trip_amount,
    p.avg_trip_amount,

    ph.peak_pickup_hour_24,
    ph.peak_pickup_hour_label,
    ph.peak_hour_trip_volume,

    p.adverse_weather_pickups,
    p.clear_or_cloudy_pickups,

    -- Percentage of classified pickups (adverse + clear/cloudy) that occurred in adverse weather.
    -- Trips with an unclassified weather condition (e.g. Fog, Unknown) are excluded from the denominator.
    ROUND(
        100.0 * p.adverse_weather_pickups /
        NULLIF(
            p.adverse_weather_pickups + p.clear_or_cloudy_pickups,
            0
        ),
        2
    ) AS adverse_weather_pickup_share_pct

FROM pickup_metrics AS p

FULL OUTER JOIN dropoff_metrics AS d
    ON p.taxi_zone_key <=> d.taxi_zone_key

LEFT JOIN peak_hour AS ph
    ON p.taxi_zone_key <=> ph.taxi_zone_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS z
    ON COALESCE(p.taxi_zone_key, d.taxi_zone_key) = z.taxi_zone_key

ORDER BY
    COALESCE(p.pickup_trip_volume, 0) DESC,
    COALESCE(p.total_trip_amount, 0) DESC;


## Business Analytics creation complete

The notebook creates Taxi Demand, Weather Behavior, and Area Mobility views in dependency-safe order. `tests/analytics/01_analytics_validation.ipynb` independently verifies availability, declared grains, valid clock hours, and reconciliation with the in-scope Gold fact population.